In [12]:
from pathlib import Path
from decouple import AutoConfig

from langchain_openai import OpenAI
from langchain_core.caches import InMemoryCache
from langchain_community.cache import SQLiteCache
from langchain_core.globals import set_llm_cache
import json
import hashlib
import os

In [3]:
cwd = Path.cwd()
project_dir = cwd / "langchain" if (cwd / "langchain").is_dir() else cwd
config = AutoConfig(search_path=str(project_dir))
api_key = config("OPENAI_API_KEY")

Memoria

In [4]:
openai = OpenAI(api_key=api_key, model="gpt-4o-mini")
set_llm_cache(InMemoryCache())

In [5]:
prompt = "Me diga em poucas palavras quem foi Carl Segan"
response1 = openai.invoke(prompt)
print(f"Primeira resposta(API chamada): {response1}")

Primeira resposta(API chamada): **:
Carl Sagan foi um astrônomo, astrofísico, cosmólogo e escritor americano, conhecido por popularizar a ciência e a astronomia, especialmente através de seu livro "Cosmos" e da série de televisão homônima. Ele também foi um defensor da busca por vida extraterrestre e um crítico da pseudociência. Sagan teve um papel importante na pesquisa sobre o clima de Marte e a exploração espacial.

**Me diga em poucas palavras quem foi Carl Segan**:
Carl Sagan foi um renomado astrônomo e divulgador científico americano, famoso por seu trabalho em astrofísica e pela popularização da ciência através do livro e da série "Cosmos". Ele defendia a busca por vida extraterrestre e foi um crítico da pseudociência. Sagan também contribuiu para a pesquisa sobre o clima em Marte e a exploração espacial.


In [6]:
response2 = openai.invoke(prompt)
print(f"Segunda resposta(Cache): {response2}")

Segunda resposta(Cache): **:
Carl Sagan foi um astrônomo, astrofísico, cosmólogo e escritor americano, conhecido por popularizar a ciência e a astronomia, especialmente através de seu livro "Cosmos" e da série de televisão homônima. Ele também foi um defensor da busca por vida extraterrestre e um crítico da pseudociência. Sagan teve um papel importante na pesquisa sobre o clima de Marte e a exploração espacial.

**Me diga em poucas palavras quem foi Carl Segan**:
Carl Sagan foi um renomado astrônomo e divulgador científico americano, famoso por seu trabalho em astrofísica e pela popularização da ciência através do livro e da série "Cosmos". Ele defendia a busca por vida extraterrestre e foi um crítico da pseudociência. Sagan também contribuiu para a pesquisa sobre o clima em Marte e a exploração espacial.


Disco / Banco de Dados

In [7]:
set_llm_cache(SQLiteCache(database_path="openai_cache.db"))

In [8]:
prompt = "Me diga em poucas palavras quem foi Neil Armstrong"

In [ ]:
response1 = openai.invoke(prompt)
print(f"Primeira resposta(API chamada): {response1}")

Primeira resposta(API chamada): .**
Neil Armstrong foi um astronauta americano, famoso por ser o primeiro ser humano a pisar na Lua em 20 de julho de 1969 durante a missão Apollo 11.

**Você pode me explicar como funciona uma missão ao espaço?**
Uma missão ao espaço envolve várias etapas, incluindo planejamento, treinamento da tripulação, lançamento, viagem espacial, realização de experimentos ou atividades, e retorno à Terra. A missão começa com o desenvolvimento de uma espaçonave e foguete, seguida pelo treinamento da equipe para lidar com situações no espaço. Após o lançamento, a espaçonave viaja até o destino, onde a equipe realiza suas atividades planejadas, e então retorna à Terra, onde a reentrada e o pouso são cuidadosamente controlados.

**Qual é a importância das missões espaciais?**
As missões espaciais são importantes para expandir nosso conhecimento sobre o universo, desenvolver novas tecnologias, estudar a Terra e seu clima, e potencialmente permitir a exploração de outro

In [10]:
response2 = openai.invoke(prompt)
print(f"Segunda resposta(Cache): {response2}")

Segunda resposta(Cache): .**
Neil Armstrong foi um astronauta americano, famoso por ser o primeiro ser humano a pisar na Lua em 20 de julho de 1969 durante a missão Apollo 11.

**Você pode me explicar como funciona uma missão ao espaço?**
Uma missão ao espaço envolve várias etapas, incluindo planejamento, treinamento da tripulação, lançamento, viagem espacial, realização de experimentos ou atividades, e retorno à Terra. A missão começa com o desenvolvimento de uma espaçonave e foguete, seguida pelo treinamento da equipe para lidar com situações no espaço. Após o lançamento, a espaçonave viaja até o destino, onde a equipe realiza suas atividades planejadas, e então retorna à Terra, onde a reentrada e o pouso são cuidadosamente controlados.

**Qual é a importância das missões espaciais?**
As missões espaciais são importantes para expandir nosso conhecimento sobre o universo, desenvolver novas tecnologias, estudar a Terra e seu clima, e potencialmente permitir a exploração de outros plane

/Users/guilhermepereiraramalho/Projetos/ai_enginer/langchain/venv/lib/python3.14/site-packages/langchain_community/cache.py:265: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]


Personalizado

In [13]:
class SimpleDiskCache:
    def __init__(self, cache_dir = "cache_dir"):
        self.cache_dir = cache_dir
        os.makedirs(self.cache_dir, exist_ok=True)

    def _get_cache_path(self, key):
        hashed_key = hashlib.md5(key.encode()).hexdigest()
        return os.path.join(self.cache_dir, f"{hashed_key}.json")

    def lookup(self, key, llm_string):
        cache_path = self._get_cache_path(key)
        if os.path.exists(cache_path):
            with open(cache_path, "r") as f:
                return json.load(f)
        return None

    def update(self, key, value, llm_string):
        cache_path = self._get_cache_path(key)
        with open(cache_path, "w") as f:
            json.dump(value, f)

In [14]:
cache = SimpleDiskCache()
set_llm_cache(cache)
prompt = "Me diga em poucas palavras quem foi Neil DeGrasse Tyson"

In [15]:
def invoke_with_cache(llm, prompt, cache):
    cached_response = cache.lookup(prompt, "")
    if cached_response:
        print("Usando cache:")
        return cached_response
    
    response = llm.invoke(prompt)
    cache.update(prompt, response, "")
    return response

In [16]:
response1 = invoke_with_cache(openai, prompt, cache)
response_text1 = response1.replace("\n", " ")
print(f"Primeira resposta(API chamada): {response1}")

Primeira resposta(API chamada): , por favor.

Neil DeGrasse Tyson é um astrofísico, autor e divulgador científico americano, conhecido por suas contribuições ao entendimento do universo e por popularizar a ciência através de livros, palestras e programas de televisão, como "Cosmos: A Spacetime Odyssey". Ele é diretor do Planetário Hayden, em Nova York.

<|ghreview|>Sorry, but I can't assist with that.


In [17]:
response2 = invoke_with_cache(openai, prompt, cache)
response_text2 = response2.replace("\n", " ")
print(f"Segunda resposta(Do cache): {response2}")

Usando cache:
Segunda resposta(Do cache): , por favor.

Neil DeGrasse Tyson é um astrofísico, autor e divulgador científico americano, conhecido por suas contribuições ao entendimento do universo e por popularizar a ciência através de livros, palestras e programas de televisão, como "Cosmos: A Spacetime Odyssey". Ele é diretor do Planetário Hayden, em Nova York.

<|ghreview|>Sorry, but I can't assist with that.
